# Greffier, le modele : 01. Exploration des donnees (M-POPP)

Premiere brique du projet. Objectif : **regarder les vraies donnees avant tout entrainement**,
pour comprendre a quoi ressemblent les actes et comment les entites sont annotees.

Principe d'honnetete tenu tout le long du projet :
- jeux train / validation / test **separes par page** (le dataset fournit deja ce decoupage),
- tous les scores annonces viendront du **jeu de test**, jamais vu a l'entrainement,
- on montre aussi les echecs, on ne choisit pas les beaux exemples.

Dataset : M-POPP (projet EXO-POPP, INSA Rouen), actes de mariage parisiens 1880-1940,
licence CC-BY 4.0. https://zenodo.org/records/10980636

## 0. Le depot (pour reutiliser outils.py)
Les notebooks partagent leurs fonctions via `modele/outils.py` du depot GitHub. On le clone
une fois, comme ca la lecture des labels et le calcul des metriques sont les memes partout.

In [ ]:
import os, sys, json, random, collections
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)  # reproductibilite

# Clone du depot pour importer modele/outils.py (ignore si deja present).
if not (Path('IA-Application') / 'modele' / 'outils.py').exists() and not Path('modele/outils.py').exists():
    os.system('git clone --depth 1 https://github.com/Abdellah427/Greffier.git IA-Application')
for cand in ('IA-Application', '.'):
    if (Path(cand) / 'modele' / 'outils.py').exists():
        sys.path.insert(0, cand)
        break
from modele.outils import charge_mpopp, extrait_entites_mpopp, texte_propre_mpopp, charge_config
CONFIG = charge_config()
print('Entites cibles :', CONFIG['entites_cibles'])

## 1. Telechargement
L'archive fait environ 1 Go. On ne la retelecharge pas si elle est deja la.

In [ ]:
RACINE = Path('m-popp')
RACINE.mkdir(exist_ok=True)
ZIP = RACINE / 'm-popp_datasets.zip'
URL = CONFIG['datasets']['m_popp']['url']
if not ZIP.exists():
    print('Telechargement...')
    os.system(f'wget -q --show-progress -O "{ZIP}" "{URL}"')
else:
    print('Deja telecharge :', ZIP)

if not (RACINE / 'extrait').exists():
    os.system(f'unzip -q -o "{ZIP}" -d "{RACINE / "extrait"}"')
print('Contenu decompresse dans', RACINE / 'extrait')

## 2. Structure des fichiers
On regarde l'arborescence pour voir l'organisation (manuscrit / imprime, images, labels).

In [ ]:
base = RACINE / 'extrait'
for chemin in sorted(base.rglob('*')):
    prof = len(chemin.relative_to(base).parts)
    if prof <= 3 and chemin.is_dir():
        print('  ' * (prof - 1) + '[' + chemin.name + ']')

print('\n--- Comptage par extension ---')
ext = collections.Counter(p.suffix.lower() for p in base.rglob('*') if p.is_file())
for e, n in ext.most_common():
    print(f'{e or "(sans)":8} {n}')

## 3. Comment les labels sont vraiment ranges
Point important, decouvert en ouvrant les fichiers : **il n'y a pas un label par image**.
Tout est dans un seul JSON, `handwritten/labels/labels-handwritten-encoding-2.json`, dont la
racine `ground_truth` est **deja decoupee en train / valid / test**. `charge_mpopp` (dans
outils.py) renvoie directement ce dictionnaire `{split: {nom_image: {"text": ...}}}`.

In [ ]:
gt = charge_mpopp(base, encodage=CONFIG['datasets']['m_popp']['encodage_retenu'])
print('Splits fournis par le dataset (separation par page) :')
for split, pages in gt.items():
    print(f'  {split:6} : {len(pages)} pages')
print('\nExemple de cle image :', list(gt['train'])[0])

## 4. Un exemple : image + annotation brute
On prend une page de test, on l'affiche, et on lit son annotation **telle quelle**. Les entites
sont encodees par des **emojis apres chaque mot** : `Charles👨💬` = epoux + prenom,
`Itsweire👨🗨` = epoux + nom, `Henri👨👴💬` = pere de l'epoux + prenom,
`Paris📖🌇` = lieu de l'acte.

In [ ]:
# Les images portent le meme nom que la cle de label.
nom = list(gt['test'])[0]
chemins_img = [p for p in base.rglob(nom) if p.is_file()]
print('Image :', nom, '->', chemins_img[0] if chemins_img else 'IMAGE INTROUVABLE')
if chemins_img:
    img = Image.open(chemins_img[0])
    plt.figure(figsize=(9, 12)); plt.imshow(img); plt.axis('off')
    plt.title(nom, fontsize=9); plt.show()

texte_brut = gt['test'][nom]['text']
print('\n--- ANNOTATION BALISEE (200 premiers caracteres) ---')
print(texte_brut[:200])

## 5. Du texte balise vers nos entites
`texte_propre_mpopp` retire les emojis (c'est la reference pour le CER), et
`extrait_entites_mpopp` traduit les balises vers notre schema commun a 12 entites
(epoux / epouse prenom, nom, metier ; peres et meres ; date ; lieu).

In [ ]:
print('--- TEXTE BRUT (reconnaissance) ---')
print(texte_propre_mpopp(texte_brut)[:300], '...\n')
print('--- ENTITES EXTRAITES (extraction d\'information) ---')
for typ, val in extrait_entites_mpopp(texte_brut):
    print(f'  {typ:16} = {val}')

## 6. Statistiques d'entites sur tout le jeu
Combien de fois chaque entite apparait, sur l'ensemble des pages. Ca confirme que le schema
tient et montre le desequilibre (le metier est plus rare que le prenom, par exemple).

In [ ]:
compte = collections.Counter()
pages_par_split = {}
for split, pages in gt.items():
    pages_par_split[split] = len(pages)
    for donnee in pages.values():
        for typ, _ in extrait_entites_mpopp(donnee['text']):
            compte[typ] += 1
print('Pages :', pages_par_split, '  total', sum(pages_par_split.values()))
print('\nOccurrences par entite :')
for typ in CONFIG['entites_cibles']:
    print(f'  {typ:16} : {compte.get(typ, 0)}')

## Ce que ce notebook fixe pour la suite
- Le format est compris : **texte de page entiere** avec entites en emojis, donc on vise un
  **petit modele vision-langage** (Florence-2 ou Qwen2-VL-2B) qui lit la page et sort le texte
  balise, pas un OCR ligne a ligne (TrOCR ne conviendrait pas).
- Le **schema a 12 entites** est confirme et code dans `outils.py` (`extrait_entites_mpopp`).
- Le **split par page** est fourni par le dataset : aucune fuite a craindre.

Prochaines etapes :
1. Notebook 02 : **Esposalles** (Espagne) et **harmonisation** vers ce meme jeu d'entites.
2. Notebook 03 : **generateur d'actes synthetiques** (FR + ES) pour entrainer a grande echelle.
3. Notebook 04 : pre-entrainement synthetique puis fine-tuning sur le reel.
4. Notebook 05 : comparaison honnete contre un gros generaliste, meme jeu de test.